# Results plotting

***

**Author:** Juan M. Montes-Sánchez

**GitHub:** https://github.com/juamonsan

***

In this notebook you will find a step by step guide where you will:
- Read a merged results .csv report file created with notebook 6 from this project.
- Plot several useful graphs for analisys purposes.

In [1]:
#----------------------------------------------------------------------
#IMPORTS
#----------------------------------------------------------------------
import matplotlib.pyplot as plt #for plotting raw data
import os #for system folders and files management
import pathlib #for system folders and files management
import pandas as pd #for Dataframe handling

from IPython.display import display # for folder selection dialog
from ipyfilechooser import FileChooser # for folder selection dialog

import ipywidgets as widgets
from ipywidgets import interact, Dropdown #for friendly user selection of different plot options

import seaborn #for plots

import numpy as np
import math
import csv

In [2]:
#----------------------------------------------------------------------
# DEFINITIONS
#----------------------------------------------------------------------





#----------------------------------------------------------------------
# DEFINITIONS END
#----------------------------------------------------------------------

## 1. Select and import CSV file

Select a merged results report in .csv format created with notebook 6 using the widget from the code below:

In [3]:
# Create and display a FileChooser widget
fc1 = FileChooser('./')
fc1.show_only_dirs=False
# Set a file filter pattern (uses https://docs.python.org/3/library/fnmatch.html)
fc1.filter_pattern = '*.csv'

print("--------------------\nSELECT CSV REPORT FILE\n--------------------")
display(fc1)
print("--------------------")

--------------------
SELECT CSV REPORT FILE
--------------------


FileChooser(path='D:\GIT\STEVAL-MKSBOX1V1_Dataset_creator', filename='', title='', show_hidden=False, select_d…

--------------------


In [4]:
print("--------------------")
print('Importing selected file ({}) into Python as a Pandas dataframe...'.format(fc1.selected))
print("--------------------")
df=pd.read_csv(fc1.selected, sep=',')
df

--------------------
Importing selected file (D:\OneDrive - US - PSI\OneDrive - UNIVERSIDAD DE SEVILLA\Doctorado\2022-23\Estancia 2023\Artículo\Results\MERGED_RESULTS_20240221_165358.csv) into Python as a Pandas dataframe...
--------------------


,Source File,Sensor Type,Sensor Name,Frequency,RNN Type,Layers,Nodes 1st Layer,Nodes 2nd Layer,Trainable Param,Non Trainable Param,Total Param,F1 Overall Accuracy,F1 Macro Average,F1 Weighted Average
0,Results_20231014_000059,ACC,LIS2DW12,1600,GRU,1,2,0,57,6,63,0.972313,0.973908,0.972228
1,Results_20231014_000059,ACC,LIS2DW12,1600,GRU,1,4,0,129,6,135,0.856678,0.859568,0.854502
2,Results_20231014_000059,ACC,LIS2DW12,1600,GRU,1,8,0,345,6,351,0.923453,0.924663,0.923597
3,Results_20231014_000059,ACC,LIS2DW12,1600,GRU,1,16,0,1065,6,1071,0.967427,0.969269,0.967300
4,Results_20231014_000059,ACC,LIS2DW12,1600,GRU,1,32,0,3657,6,3663,0.964169,0.966168,0.964007
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4889,Results_models_reduced_20240131_193101,ACC,LSM6DSOX,416,GRU,2,4,2,171,6,177,0.657895,0.561297,0.540913
4890,Results_models_reduced_20240131_193101,ACC,LSM6DSOX,416,LSTM,2,4,2,199,6,205,0.654605,0.558065,0.537793
4891,Results_models_reduced_20240131_193101,ACC,LSM6DSOX,52,LSTM,2,4,2,199,6,205,0.773026,0.759850,0.746878
4892,Results_models_reduced_20240131_193101,ACC,LSM6DSOX,833,GRU,2,4,2,171,6,177,0.657895,0.561297,0.540913


In [5]:
# Combine RNN and layers in a new column, for easier plotting afterwards
df['RNN'] = df['RNN Type'].astype(str) + ' ' + df['Layers'].astype(str) + ' layers'

# Add units to frequency
#df['Frequency'] = df['Frequency'].astype(str) + ' Hz'

## 2. Plot 

In [9]:
def print_sensor(Type, Name):
    print ("Plot for {} {}".format(Type, Name))
    df2=df.loc[(df['Sensor Type'] == Type) & (df['Sensor Name'] == Name)]
    
    #Savepath for figure file (same as CSV file location)
    FIGpath=os.path.join(fc1.selected_path,"{} {}.svg".format(Type, Name))
    
    # Grouping not necessary with seaborn...
    #df2_grouped=df2.groupby(['RNN Type','Layers'])
    
    # Plot using seaborn (https://seaborn.pydata.org/generated/seaborn.catplot.html)
    # idea from https://stackoverflow.com/questions/31845258/multi-index-plotting
    fg = seaborn.catplot(x='Nodes 1st Layer', y='F1 Weighted Average', hue='RNN', 
                        col='Frequency', data=df2, kind='bar', errorbar=("pi", 100))
    fg.set_xlabels('Number of nodes in layer 1')
    fg.fig.suptitle("{} {}".format(Type, Name))
    fg.savefig(FIGpath, format="svg")
    
# -------------------------------------------------------------   
# Failed plotting, not removed yet for improving purposes:
# -------------------------------------------------------------    
#     fig, axes = plt.subplots(1, len(df2_grouped), sharey=True, figsize=(20,4))
#     for ax, (Frequency, group) in zip(axes, df2_grouped):
#         # The rename_axis function makes the difference
#         group.set_index('Nodes 1st Layer').rename_axis()["F1 Weighted Average"].plot(kind="line", ax=ax)
#         ax.tick_params(axis='both', which='both', length=0)

#     ax.legend()
#     fig.subplots_adjust(wspace=0)
    
#     df2_grouped['F1 Weighted Average'].plot(kind='line', 
#                                             title="{} {}".format(Type, Name), 
#                                             figsize=(10,5),
#                                             legend=True, x=['Frequency','Nodes 1st Layer'])
# -------------------------------------------------------------



    
def select_type(Type):
    sensorW.options = df['Sensor Name'].loc[df['Sensor Type'].eq(Type)].unique().tolist()


#A list with all unique sensor types:
Types = df['Sensor Type'].unique().tolist()

#A widget for selecting one of the unique types:
sensortypeW = widgets.Dropdown(options=Types)
SelectedType = sensortypeW.value

#A list with all unique sensor names which type is SelectedType:
Sensors=df['Sensor Name'].loc[df['Sensor Type'].eq(SelectedType)].unique().tolist()

#A widget for selecting one of the unique sensors which type is SelectedType:
sensorW = widgets.Dropdown(options=Sensors)
SelectedName = sensorW.value

#Display interactive selection widgets:
i = widgets.interactive(print_sensor, Type=sensortypeW, Name=sensorW)
widgets.interactive(select_type, Type=sensortypeW)
display(i)

#Plots F1 from selected sensor
#df2=df.loc[(df['Sensor Type'] == SelectedType) & (df['Sensor Name'] == SelectedName)]
#df2.plot()

interactive(children=(Dropdown(description='Type', options=('ACC', 'GYRO', 'MAG'), value='ACC'), Dropdown(desc…

## 3. Network complexity

In [8]:
# Keep only one value for each different configuration (all RNN of the same type has same number of param)
df_comp=df.loc[df.groupby(['RNN', 'Nodes 1st Layer'])['Total Param'].idxmin()]
df_comp

,Source File,Sensor Type,Sensor Name,Frequency,RNN Type,Layers,Nodes 1st Layer,Nodes 2nd Layer,Trainable Param,Non Trainable Param,Total Param,F1 Overall Accuracy,F1 Macro Average,F1 Weighted Average,RNN
0,Results_20231014_000059,ACC,LIS2DW12,1600,GRU,1,2,0,57,6,63,0.972313,0.973908,0.972228,GRU 1 layers
1703,Results_20231020_125409,GYRO,LSM6DSOX,416,GRU,1,4,0,57,6,63,0.656250,0.559996,0.539705,GRU 1 layers
1704,Results_20231020_125409,GYRO,LSM6DSOX,416,GRU,1,8,0,129,6,135,0.684211,0.616814,0.598342,GRU 1 layers
1705,Results_20231020_125409,GYRO,LSM6DSOX,416,GRU,1,16,0,345,6,351,0.667763,0.584682,0.565063,GRU 1 layers
1706,Results_20231020_125409,GYRO,LSM6DSOX,416,GRU,1,32,0,1065,6,1071,0.574013,0.512243,0.498443,GRU 1 layers
1707,Results_20231020_125409,GYRO,LSM6DSOX,416,GRU,1,64,0,3657,6,3663,0.634868,0.577750,0.556659,GRU 1 layers
6,Results_20231014_000059,ACC,LIS2DW12,1600,GRU,2,2,1,69,6,75,0.657980,0.561245,0.541055,GRU 2 layers
1709,Results_20231020_125409,GYRO,LSM6DSOX,416,GRU,2,4,2,69,6,75,0.656250,0.559996,0.539705,GRU 2 layers
1710,Results_20231020_125409,GYRO,LSM6DSOX,416,GRU,2,8,4,171,6,177,0.664474,0.575270,0.555371,GRU 2 layers
1711,Results_20231020_125409,GYRO,LSM6DSOX,416,GRU,2,16,8,501,6,507,0.671053,0.600069,0.580961,GRU 2 layers


## 4. Best results 

In [6]:
# Keep only maximum values for each different configuration (F1 Weighted Average)
dfMAX=df.loc[df.groupby(['Sensor Type', 
                         'Sensor Name',
                         'Frequency',
                         'RNN Type',
                         'Layers',
                         'Nodes 1st Layer'])['F1 Weighted Average'].idxmax()]
dfMAX

,Source File,Sensor Type,Sensor Name,Frequency,RNN Type,Layers,Nodes 1st Layer,Nodes 2nd Layer,Trainable Param,Non Trainable Param,Total Param,F1 Overall Accuracy,F1 Macro Average,F1 Weighted Average,RNN
984,Results_20231020_125409,ACC,LIS2DW12,12,GRU,1,2,0,57,6,63,0.656352,0.559362,0.539426,GRU 1 layers
2855,Results_20240202_160311,ACC,LIS2DW12,12,GRU,1,4,0,129,6,135,0.669381,0.590829,0.572042,GRU 1 layers
2856,Results_20240202_160311,ACC,LIS2DW12,12,GRU,1,8,0,345,6,351,0.757329,0.748234,0.734651,GRU 1 layers
51,Results_20231014_000059,ACC,LIS2DW12,12,GRU,1,16,0,1065,6,1071,0.783388,0.780498,0.768445,GRU 1 layers
2858,Results_20240202_160311,ACC,LIS2DW12,12,GRU,1,32,0,3657,6,3663,0.789902,0.789050,0.777198,GRU 1 layers
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4601,Results_20240219_235305,MAG,LIS2MDL,100,LSTM,2,4,2,199,6,205,0.998369,0.998472,0.998369,LSTM 2 layers
2730,Results_20231023_192400,MAG,LIS2MDL,100,LSTM,2,8,4,613,6,619,1.000000,1.000000,1.000000,LSTM 2 layers
1795,Results_20231020_125409,MAG,LIS2MDL,100,LSTM,2,16,8,613,6,619,1.000000,1.000000,1.000000,LSTM 2 layers
1796,Results_20231020_125409,MAG,LIS2MDL,100,LSTM,2,32,16,2113,6,2119,1.000000,1.000000,1.000000,LSTM 2 layers


In [8]:
#Save as CSV file
#Savepath same as source CSV file location
CSVpath=os.path.join(fc1.selected_path,"MAX_RESULTS.csv")
dfMAX.to_csv(path_or_buf=CSVpath,header=True,index=False)


In [12]:
#Graph best 10
F1_Theresold=0.95;
SensorType='ACC'

dfSorted=dfMAX.loc[(dfMAX['Sensor Type']==SensorType) & (dfMAX['F1 Weighted Average']>=0.95)]
dfSorted=dfSorted.sort_values(['Frequency','Total Param'], ascending=[True, True], ignore_index=True)
dfSorted.head(10)

#Savepath for figure file (same as CSV file location)
FIG2path=os.path.join(fc1.selected_path,"TOP 10 {} F1 {}.svg".format(SensorType, F1_Theresold))

,Source File,Sensor Type,Sensor Name,Frequency,RNN Type,Layers,Nodes 1st Layer,Nodes 2nd Layer,Trainable Param,Non Trainable Param,Total Param,F1 Overall Accuracy,F1 Macro Average,F1 Weighted Average,RNN
0,Results_20231023_192400,ACC,LIS3DHH,68,LSTM,1,8,0,417,6,423,0.952692,0.955263,0.952358,LSTM 1 layers
1,Results_20231023_192400,ACC,LIS3DHH,68,LSTM,1,16,0,1337,6,1343,0.952692,0.955263,0.952358,LSTM 1 layers
2,Results_20231023_192400,ACC,LIS3DHH,68,GRU,2,16,8,1665,6,1671,0.955954,0.958388,0.955678,GRU 2 layers
3,Results_20240219_235305,ACC,LIS3DHH,68,GRU,1,32,0,3657,6,3663,0.951060,0.953698,0.950695,GRU 1 layers
4,Results_20231023_192400,ACC,LIS3DHH,68,LSTM,1,32,0,4713,6,4719,0.954323,0.956827,0.954018,LSTM 1 layers
5,Results_20231023_192400,ACC,LIS3DHH,68,GRU,2,32,16,6009,6,6015,0.952692,0.955263,0.952358,GRU 2 layers
6,Results_20231020_125409,ACC,LIS3DHH,68,LSTM,2,32,16,7801,6,7807,0.955954,0.958388,0.955678,LSTM 2 layers
7,Results_20231014_000059,ACC,LIS3DHH,68,GRU,1,64,0,13449,6,13455,0.962480,0.964616,0.962297,GRU 1 layers
8,Results_20240219_235305,ACC,LIS3DHH,68,LSTM,1,64,0,17609,6,17615,0.955954,0.958426,0.955710,LSTM 1 layers
9,Results_20231020_125409,ACC,LIS3DHH,68,GRU,2,64,32,22761,6,22767,0.959217,0.961506,0.958991,GRU 2 layers
